<a href="https://colab.research.google.com/github/TBGhorbanpour/Social-Awareness/blob/main/Community_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install networkx python-louvain -q

In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 44.1 MB/s eta 0:00:00


In [3]:
!pip install hazm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.9 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9371 sha256=a175bcf9733bc258380f79028762a1d322e2c7bdc013722d9c8c946fa8ca8899
  Stored in directory: /root/.cache/pip/wheels/be/12/c9/228313ff5cb722777830302f1d4136de4129932e6a0354c056
Successfully built flashtext


In [4]:
import os
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
INPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_Sentiment_Output.csv'
data = pd.read_csv(INPUT_CSV)

In [6]:
import os
OUTPUT_DIR =  '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
# --- PART A: REBUILD VARIABLES & MOUNT DRIVE ---
import pandas as pd
import re
import unicodedata
from sklearn.feature_extraction.text import CountVectorizer
from hazm import stopwords_list
from google.colab import drive
import os

# 2. Recreate Stopwords
persian_stopwords = set(stopwords_list())
custom_stopwords = {
    'و', 'در', 'به', 'از', 'که', 'می', 'این', 'است', 'را', 'با', 'های', 'ی',
    'برای', 'تا', 'اما', 'چه', 'آیا', 'هم', 'نه', 'بله', 'یک', 'دو', 'سه',
    'من', 'تو', 'او', 'ما', 'شما', 'آنها', 'کجا', 'کی', 'چرا', 'چگونه',
    'دل', 'قیمت', 'دنیا', 'گران', 'ربط', 'پول', 'کشور', 'خبر', 'عزیز',
    'دوست', 'یاد', 'کار', 'فیلم', 'موضوع', 'خرید', 'شب', 'ماه', 'قشنگ',
    'گوش', 'گل', 'زیبا', 'خوار', 'مرغ', 'ماده', 'جهان', 'زندگی', 'اروپا',
    'ماشین', 'ادامه', 'هفته'
}
final_stopwords = list(persian_stopwords.union(custom_stopwords))

# 3. Recreate Normalization Function
def fast_normalize(text):
    if not isinstance(text, str): return ""
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('ي', 'ی').replace('ك', 'ک').replace('ة', 'ه')
    text = re.sub(r'http\S+|www\.\S+|@\S+|#', '', text)
    text = re.sub(r'[0-9۰-۹]', '', text)
    text = re.sub(r'[^\w\s\u200c]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# 4. Rebuild clean_tweets, vectorizer, and X
print("Rebuilding text variables...")
# Make sure 'data' is loaded in your notebook before running this!
clean_tweets = data['lemmatized_tweet'].astype(str).apply(fast_normalize)

vectorizer = CountVectorizer(
    token_pattern=r'(?u)\b\w\w+\b',
    stop_words=final_stopwords,
    max_features=10000
)
X = vectorizer.fit_transform(clean_tweets)
print("✅ Variables rebuilt successfully!\n")

Rebuilding text variables...


/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['آید', 'توان', 'تواند', 'توانند', 'رسد', 'رود', 'سال', 'نمی', 'گوید', 'گویند'] not in stop_words.
  warnings.warn(


✅ Variables rebuilt successfully!



In [8]:
import networkx as nx
import community as community_louvain
import pandas as pd
import itertools
from collections import Counter
import gensim.corpora as corpora
from gensim.models import CoherenceModel

print("Step 1: Building OPTIMIZED Word Co-occurrence Network...")

# 1. Get top 2000 words
feature_names = vectorizer.get_feature_names_out()
word_counts = X.sum(axis=0).A1
top_word_indices = word_counts.argsort()[-2000:][::-1]
allowed_words = set(feature_names[top_word_indices])

G = nx.Graph()

# 2. Build edges WITH A WEIGHT THRESHOLD (Crucial for lowering conductance!)
MIN_COOCCURRENCE = 3 # Words must appear together at least 3 times to form an edge

for text in clean_tweets:
    words = text.split()
    unique_words = list(set([w for w in words if w in allowed_words]))

    if len(unique_words) > 1:
        for w1, w2 in itertools.combinations(unique_words, 2):
            if G.has_edge(w1, w2):
                G[w1][w2]['weight'] += 1
            else:
                G.add_edge(w1, w2, weight=1)

# Remove weak edges
weak_edges = [(u, v) for (u, v, d) in G.edges(data=True) if d['weight'] < MIN_COOCCURRENCE]
G.remove_edges_from(weak_edges)

# Remove isolated nodes (words with no strong connections left)
G.remove_nodes_from(list(nx.isolates(G)))

print(f"Optimized Graph built! Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

print("Step 2: Testing Multiple Resolutions to find the optimum...")

# Prepare texts for coherence calculation
texts = [doc.split() for doc in clean_tweets]
dictionary = corpora.Dictionary(texts)

# Test different resolutions
resolutions_to_test = [0.8, 1.0, 1.2, 1.5, 2.0]
best_result = None
best_avg_cv = -1.0

results_log = []

for res in resolutions_to_test:
    partition = community_louvain.best_partition(G, weight='weight', resolution=res, random_state=100)
    mod_q = community_louvain.modularity(partition, G)

    # Group words by community
    comm_words_dict = {}
    for word, comm_id in partition.items():
        comm_words_dict.setdefault(comm_id, []).append(word)

    valid_comms = 0
    total_cv = 0.0

    for comm_id, words in comm_words_dict.items():
        # FILTER: Only evaluate communities between 15 and 200 words
        if 15 <= len(words) <= 200:
            try:
                cm = CoherenceModel(topics=[words], texts=texts, dictionary=dictionary, coherence='c_v')
                cv = cm.get_coherence()
                total_cv += cv
                valid_comms += 1
            except:
                pass

    avg_cv = total_cv / valid_comms if valid_comms > 0 else 0.0
    num_comms = len(comm_words_dict)

    results_log.append({
        'Resolution': res,
        'Num_Communities': num_comms,
        'Valid_Communities (15-200 words)': valid_comms,
        'Modularity_Q': round(mod_q, 4),
        'Avg_C_v_Score': round(avg_cv, 4)
    })

    if avg_cv > best_avg_cv and valid_comms > 3: # Ensure we have enough valid communities
        best_avg_cv = avg_cv
        best_result = {
            'resolution': res,
            'partition': partition,
            'modularity': mod_q,
            'comm_words_dict': comm_words_dict
        }

# Display the comparison table
comparison_df = pd.DataFrame(results_log)
print("\n--- RESOLUTION COMPARISON ---")
display(comparison_df)

print(f"\n✅ BEST CONFIGURATION FOUND: Resolution = {best_result['resolution']}")

# --- STEP 3: SAVE THE BEST RESULTS ---
print("Step 3: Calculating detailed metrics for the BEST configuration and saving...")

final_partition = best_result['partition']
final_comm_dict = best_result['comm_words_dict']

# Calculate Conductance for the best partition
def calc_conductance(G, partition):
    communities = {}
    for node, comm_id in partition.items():
        communities.setdefault(comm_id, set()).add(node)

    results = []
    total_vol = sum(dict(G.degree()).values())
    for comm_id, nodes in communities.items():
        vol_c = sum(G.degree(n) for n in nodes)
        vol_rest = total_vol - vol_c
        cut_size = sum(G[u][v]['weight'] for u in nodes for v in G[u] if v not in nodes)
        cond = cut_size / min(vol_c, vol_rest) if min(vol_c, vol_rest) > 0 else 0.0
        results.append({'Community_ID': comm_id, 'Conductance': round(cond, 4), 'Num_Words': len(nodes)})
    return pd.DataFrame(results).sort_values('Community_ID')

cond_df = calc_conductance(G, final_partition)

# Calculate C_v for the best partition
cv_results = []
for comm_id, words in final_comm_dict.items():
    if 15 <= len(words) <= 200: # Only keep valid ones
        try:
            cm = CoherenceModel(topics=[words], texts=texts, dictionary=dictionary, coherence='c_v')
            cv_results.append({'Community_ID': comm_id, 'C_v_Score': round(cm.get_coherence(), 4)})
        except:
            pass

cv_df = pd.DataFrame(cv_results)
final_validation_df = pd.merge(cond_df, cv_df, on='Community_ID', how='inner')

print("\n--- FINAL VALIDATED COMMUNITIES ---")
display(final_validation_df)

# Save to Google Drive
final_validation_df.to_csv(f'{OUTPUT_DIR}/final_optimized_communities.csv', index=False, encoding='utf-8-sig')

# Save the word mapping for the best partition
pd.DataFrame(list(final_partition.items()), columns=['Word', 'Community_ID']).to_csv(
    f'{OUTPUT_DIR}/final_community_mapping.csv', index=False, encoding='utf-8-sig'
)

print(f"\n✅ All optimized results saved to {OUTPUT_DIR}")

Step 1: Building OPTIMIZED Word Co-occurrence Network...
Optimized Graph built! Nodes: 1999, Edges: 199344
Step 2: Testing Multiple Resolutions to find the optimum...

--- RESOLUTION COMPARISON ---


,Resolution,Num_Communities,Valid_Communities (15-200 words),Modularity_Q,Avg_C_v_Score
0,0.8,14,5,0.1574,0.4023
1,1.0,5,0,0.1747,0.0000
2,1.2,11,6,0.1607,0.3571
3,1.5,24,18,0.1387,0.4018
4,2.0,50,32,0.1167,0.3996



✅ BEST CONFIGURATION FOUND: Resolution = 0.8
Step 3: Calculating detailed metrics for the BEST configuration and saving...

--- FINAL VALIDATED COMMUNITIES ---


,Community_ID,Conductance,Num_Words,C_v_Score
0,1,8.0542,159,0.4334
1,2,9.5866,56,0.3695
2,6,9.8573,69,0.3795
3,11,15.4654,30,0.4107
4,12,13.7342,32,0.4186



✅ All optimized results saved to /content/drive/MyDrive/Thesis/Data/Data/Data_Paper


In [9]:
import pandas as pd
import networkx as nx

# 1. Load the optimized mapping we just saved
mapping_df = pd.read_csv(f'{OUTPUT_DIR}/final_community_mapping.csv')

# 2. Calculate word importance (degree) from the optimized graph
degrees = dict(G.degree())

# 3. Group words by community and sort by importance
community_details = []

for comm_id in sorted(mapping_df['Community_ID'].unique()):
    # Get all words in this community
    words = mapping_df[mapping_df['Community_ID'] == comm_id]['Word'].tolist()

    # Sort words by their degree (number of connections) in descending order
    sorted_words = sorted(words, key=lambda w: degrees.get(w, 0), reverse=True)

    community_details.append({
        'Community_ID': comm_id,
        'Total_Words': len(sorted_words),
        'Top_15_Core_Words': " | ".join(sorted_words[:15]),
        'All_Words': ", ".join(sorted_words)
    })

# 4. Create and display the DataFrame
details_df = pd.DataFrame(community_details)

# Merge with our validation scores to see which ones are the best
final_report = pd.merge(details_df, final_validation_df, on='Community_ID', how='left')

# Sort by C_v Score descending to see the best communities first
final_report = final_report.sort_values('C_v_Score', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("FINAL COMMUNITY WORDS (Sorted by Best Coherence)")
print("="*80)

# Display only the most important columns for readability
pd.set_option('display.max_colwidth', None)
display(final_report[['Community_ID', 'C_v_Score', 'Total_Words', 'Top_15_Core_Words']])

# Save this highly readable report to your Drive
final_report.to_csv(f'{OUTPUT_DIR}/community_words_and_scores.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ Saved detailed word lists to 'community_words_and_scores.csv' in your Drive")


FINAL COMMUNITY WORDS (Sorted by Best Coherence)


,Community_ID,C_v_Score,Total_Words,Top_15_Core_Words
0,1,0.4334,159,الوده | هوا | تهران | مدیریت | وضعیت | بحران | ملی | کاهش | اداره | مرکز | اقلیم | علم | بارش | صبح | تعطیل
1,12,0.4186,32,دریاچه | ارومیه | احیا | اذربایجان | نمک | تبریز | بودجه | ستاد | مشابه | سیب | بانمک | تراز | زار | هشتگ | عملکرد
2,11,0.4107,30,منبع | طبیعی | نفت | گاز | معدن | ثروت | میراث | ملک | اختیار | غنی | تاراج | طبقه | ثروتمند | موقعیت | جغرافیایی
3,6,0.3795,69,گرد | غبار | خاک | طوفان | اسمان | نفس | رنگ | شن | قرمز | کانون | بهار | سرخ | فرسایش | خفه | نور
4,2,0.3695,56,سگ | ولگرد | حمله | خیابان | شهرداری | اکوسیستم | صاحب | زاد | وحشی | عقیم | هار | معضل | خانگی | گله | کوچه
5,0,NaN,294,زیست | محیط | جنگل | سازمان | تالاب | فعال | میانکاله | رئیس | بوم | حفاظت | نیرو | گزارش | تخریب | اتش | قانون
6,3,NaN,3,پوشش | گیاهی | عسل
7,4,NaN,4,بافت | تخت | ریزش | خاری
8,5,NaN,14,نامه | دین | مصیبت | مسجد | نمیتوان | مایه | اعظم | درب | نادان | اتحاد | بنیاد | شرم | نیشابور | اذان
9,7,NaN,848,سال | ایران | خشکسالی | قحطی | نابود | حیوان | خانه | حمایت | جنگ | غذا | دولت | جان | ایرانی | پسماند | شرایط



✅ Saved detailed word lists to 'community_words_and_scores.csv' in your Drive


3 of them are very small (between 19 and 22 words):
Community 15 (19 words),
Community 22 (22 words),
Community 13 (20 words).


---


Therefore, out of the 17 mathematically valid communities, 14 of them are large, robust, and distinct enough (ranging from 63 to 199 words) to be considered the "main" themes


---


Total clusters found by Louvain: 24
Clusters rejected for being too small (<15 words) or too large (>200 words): 7
Mathematically valid clusters remaining: 17
Robust, major themes suitable for high-level analysis: ~14

In [10]:
import pandas as pd
import os

# 1. Define the 14 Robust Community IDs
# (Excluding the tiny <20 word clusters and the massive >200 word garbage bins)
target_communities = [8, 0, 6, 10, 14, 1, 12, 23, 16, 5, 20, 7, 2, 13]

# 2. Load the community mapping from your Drive
mapping_df = pd.read_csv(f'{OUTPUT_DIR}/final_community_mapping.csv')

# Filter to only keep the 14 target communities (FIXED TYPO HERE)
filtered_mapping = mapping_df[mapping_df['Community_ID'].isin(target_communities)]

# 3. Create a fast lookup dictionary: Word -> Community_ID
word_to_comm = {}
for _, row in filtered_mapping.iterrows():
    word_to_comm[row['Word']] = row['Community_ID']

print(f"Loaded {len(word_to_comm)} unique words across 14 communities.")

# 4. Define the tagging function
def assign_community(tweet_words):
    if not isinstance(tweet_words, str):
        return -1

    words = tweet_words.split()
    comm_counts = {}

    # Count how many words in this tweet belong to each community
    for word in words:
        if word in word_to_comm:
            comm_id = word_to_comm[word]
            comm_counts[comm_id] = comm_counts.get(comm_id, 0) + 1

    # Return the community with the highest word count
    if comm_counts:
        return max(comm_counts, key=comm_counts.get)
    else:
        return -1  # -1 means "Uncategorized / No strong match"

# 5. Apply the tagging to your 75,000 tweets
print("Tagging 75,000 tweets (this will take ~10 seconds)...")
data['Community_ID_14'] = [assign_community(tweet) for tweet in clean_tweets]

# 6. View the distribution
print("\n--- Tweet Distribution across 14 Communities ---")
print(data['Community_ID_14'].value_counts().sort_index())

# 7. Save the updated dataframe to Google Drive
output_file = f'{OUTPUT_DIR}/data_tagged_with_14_communities.csv'
data.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n✅ Successfully saved tagged dataset to: {output_file}")

Loaded 1961 unique words across 14 communities.
Tagging 75,000 tweets (this will take ~10 seconds)...

--- Tweet Distribution across 14 Communities ---
Community_ID_14
-1        62
 0      6859
 1      2419
 2      2651
 5        81
 6      2458
 7     36866
 8        26
 10    21658
 12     2373
 13        2
Name: count, dtype: int64

✅ Successfully saved tagged dataset to: /content/drive/MyDrive/Thesis/Data/Data/Data_Paper/data_tagged_with_14_communities.csv


In [11]:
import pandas as pd
import networkx as nx
import community as community_louvain
from collections import Counter
import os

print("Preparing network metrics...")

# Safe fallback: Load graph and re-calculate partition if variables were cleared
if 'G' not in globals() or 'final_partition' not in globals():
    print("Loading optimized graph from Drive...")
    G = nx.read_graphml(f'{OUTPUT_DIR}/word_network.graphml')

    # Restore edge weights (GraphML saves them as strings)
    for u, v, d in G.edges(data=True):
        d['weight'] = float(d['weight'])

    # Re-run Louvain with our best found resolution (1.5)
    print("Re-calculating Louvain partition...")
    final_partition = community_louvain.best_partition(G, weight='weight', resolution=1.5, random_state=100)

# 1. Calculate Global Metrics
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
num_communities = len(set(final_partition.values()))
modularity_q = community_louvain.modularity(final_partition, G)

# 2. Calculate Size of Each Community
community_sizes = Counter(final_partition.values())
sorted_sizes = dict(sorted(community_sizes.items()))

# 3. Create the Summary DataFrame (All in one table)
summary_data = {
    'Metric': [
        'Nodes',
        'Edges',
        'Communities',
        'Q Modularity',
        'Community Sizes'
    ],
    'Value': [
        num_nodes,
        num_edges,
        num_communities,
        f"{modularity_q:.4f}",
        str(sorted_sizes) # Saved as a string dictionary to fit in one cell
    ]
}

summary_df = pd.DataFrame(summary_data)

# 4. Save to a single CSV file in Google Drive
output_path = f'{OUTPUT_DIR}/community_summary_final.csv'
summary_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print("="*60)
print("✅ Successfully saved to 'community_summary_final.csv'")
print("="*60)
print("\nPreview of the CSV:")
display(summary_df)

Preparing network metrics...
✅ Successfully saved to 'community_summary_final.csv'

Preview of the CSV:


,Metric,Value
0,Nodes,1999
1,Edges,199344
2,Communities,14
3,Q Modularity,0.1574
4,Community Sizes,"{0: 294, 1: 159, 2: 56, 3: 3, 4: 4, 5: 14, 6: 69, 7: 848, 8: 2, 9: 1, 10: 486, 11: 30, 12: 32, 13: 1}"
